# Clase 3b — Evaluación, monitoreo y benchmarks de sistemas LLM

### *Del prompt al producto: cómo saber si tu LLM funciona bien — y cómo seguir sabiéndolo.*

---

**Docentes:** Dr. Jorge Roa · Dra. Milagros Gutiérrez
**CIDISI — UTN-FRSF — IA · 2026**


## Recap Clases 1, 2 y 3 → conexión con hoy

```
  Clase 1            Clase 2              Clase 3 (RAG)              HOY (3b)
  ───────            ───────              ─────────────              ────────
  Tokens             LLMs +               Retrieval +                Evaluar
  Embeddings    →    Transformer    →     Generation        →        + Monitorear
  Vectores           Prompting            (BM25/Vector/Graph)        + Benchmarkear
                     llamar_llm()         rag_query()                el sistema
```

| Lo que ya sabemos construir | Lo que resolvemos hoy |
|---|---|
| Llamar un LLM y prompearlo (Clase 2) | ¿Cómo sé si las respuestas son **buenas**? |
| Sistemas RAG con pipeline propio (Clase 3) | ¿Cómo detecto que el retrieval o la generación **fallaron**? |
| Modelos open weight y APIs | ¿Cómo elijo entre ellos con **datos**, no con intuición? |
| Apps que funcionan en demo | ¿Cómo siguen funcionando en **producción**? |

> **Cierre de pendiente:** en Clase 2 dejamos "evaluación y monitoreo" pendiente. Hoy lo cubrimos en profundidad y con práctica end-to-end con Arize AX.


## Recorrido de la clase

```
  Fase 0          Fase 1          Fase 2        Fase 3         Fase 4
  ──────          ──────          ──────        ──────         ──────
  Fundamentos  →  Pre-deploy  →   Deploy   →    Producción  →  Cierre
  (qué medir,     (eval offline)  (release      (monitoring,   (loop
   con qué)                        patterns)     drift)         continuo)
```

| Bloque | Tema | Tiempo aprox. |
|---|---|---|
| **B1** | Fundamentos — por qué evaluar LLMs es distinto, métricas, tracing | ~15 min |
| **B2** | Pre-deploy (eval offline) — datasets, LLM-as-judge, RAGAS, safety, benchmarks | ~40 min |
| ☕ | _break_ | 10 min |
| **B3** | Deploy / release — A/B, shadow, canary, prompt versioning, LLMOps | ~15 min |
| **B4** | Producción / monitoring — logging, cost/latency, drift, feedback loops | ~15 min |
| **B5** | Cierre — el loop completo, cuándo invertir en cada cosa | ~5 min |
| 🛠 | **Práctica end-to-end con Arize AX** | ~35 min |

> El bloque cierra con una **práctica hands-on**: instrumentamos un chatbot Q&A, le corremos evals offline, lo trazamos en Arize, armamos dashboards y simulamos drift.


## ⚙️ Setup — para la práctica de B5

Esta clase es **conceptual + práctica con Arize AX**. La práctica vive en una notebook separada (Colab-friendly):

📓 [`clase03/notebooks/01_arize_eval_handson.ipynb`](notebooks/01_arize_eval_handson.ipynb)

**Antes del segmento práctico:**
1. **Cuenta gratis en Arize AX** → https://app.arize.com/ (signup con email, ~3 min)
2. **API Key de Groq** → si ya la tenés de Clase 2, listo
3. Abrir el notebook en Colab cuando lleguemos a la práctica

**Dependencias del notebook** (las instala la primera celda):
```bash
pip install arize arize-otel openinference-instrumentation-groq groq pandas matplotlib
```

> Las slides de este bloque son **todas conceptuales**. El código vive en la notebook hands-on.


# Bloque 1
## Fundamentos — qué medir, con qué

---

*Antes de elegir herramienta, conviene entender por qué evaluar LLMs no es como evaluar un clasificador clásico.*


## ¿Por qué evaluar LLMs es distinto?

| ML clásico | LLMs |
|---|---|
| Output discreto (clase, número) | Output texto libre, espacio infinito |
| Accuracy / F1 / RMSE bien definidos | "¿Está bien la respuesta?" es subjetivo |
| Test set con label único | Múltiples respuestas pueden ser correctas |
| Determinístico para mismo input | Estocástico (`temperature > 0`) |
| Falla → respuesta incorrecta | Falla → puede **inventar con confianza** (alucinar) |

> **El problema central:** no hay un "ground truth" único ni una métrica universal. Hay que **diseñar el protocolo de eval a medida** del producto.


## Reference-based vs reference-free

| Familia | Métrica | Necesita ground truth | Cuándo sirve |
|---|---|---|---|
| **Reference-based léxico** | BLEU, ROUGE, METEOR | Sí (referencia exacta) | Traducción / resumen con referencia humana fija |
| **Reference-based semántico** | BERTScore, embedding similarity | Sí (semántica) | Cuando hay paráfrasis válidas |
| **Reference-free determinista** | Exact match, regex, parseo JSON | No | Outputs estructurados (clasificación, extracción) |
| **Reference-free LLM-as-judge** | Faithfulness, relevance, quality (0-5) | No | Respuestas abiertas (chat, RAG, creativo) |

> Para LLMs en producción **reference-free domina**: no podés tener referencia humana para cada query real. Pero **reference-based sigue siendo útil offline** para regresión sobre golden datasets.


## Detección de alucinaciones (sin ground truth)

| Técnica | Idea | Pros | Contras |
|---|---|---|---|
| **SelfCheckGPT** | Generar N respuestas con `temperature>0` y medir consistencia | No necesita referencia | Caro (N veces el costo) |
| **Citation-based** | Forzar al LLM a citar fragmentos del contexto y verificar coverage | Auditable, explicable | Sólo aplica si hay contexto (RAG) |
| **Faithfulness via NLI** | Pedirle a un modelo NLI/LLM-juez si cada afirmación está soportada | Granularidad por afirmación | Depende del juez |
| **Token-level uncertainty** | Probabilidades del logits (entropy, perplejidad) | Gratis si tenés acceso | Sólo modelos open / API que lo expongan |

> Para RAG, **citation-based + faithfulness** son la base. Para chatbots sin contexto, **SelfCheckGPT** o un **juez con web search** son las opciones.

> 📖 *Manakul et al. (2023). SelfCheckGPT.* https://arxiv.org/abs/2303.08896


## Tracing y observability — fundamentos

```
  Trace = una "request" end-to-end
  ┌──────────────────────────────────────────────────┐
  │  user_query → retrieve → rerank → generate       │
  │                                                   │
  │  Span 1: retrieve     [12 ms]                     │
  │  Span 2: rerank       [340 ms]                    │
  │  Span 3: generate     [1850 ms, 432 tokens, $0.02]│
  │                                                   │
  │  Total: 2.2s · $0.02 · trace_id = abc-123         │
  └──────────────────────────────────────────────────┘
```

**Conceptos clave:**
- **Span** — operación atómica con start/end, atributos, eventos.
- **Trace** — DAG de spans de una misma request (`trace_id` los une).
- **Context propagation** — cómo el `trace_id` viaja entre componentes (HTTP headers, runtime context).
- **Attributes** — input, output, latency, tokens, costo, model, prompt_version…
- **OpenTelemetry (OTel)** — el estándar abierto. Arize, Langfuse, Phoenix lo soportan.

> Sin tracing **no hay debugging post-mortem**. En producción, "¿por qué esta respuesta fue mala?" sólo se contesta con un trace.


# Bloque 2
## Pre-deploy — evaluación offline

---

*El protocolo que corre antes de cada release: dataset → eval → score → decisión de merge.*


## El loop offline — `dataset → eval → score`

<img src="figures/evals_offline_online.svg" alt="Eval offline (pre-deploy con dataset estático en CI) vs eval online (en producción con tráfico real)" class="slide-figure" style="width: 86%;"/>

```
  Eval offline (pre-deploy)
  ─────────────────────────
  golden_dataset.jsonl  ──┐
                          ├──▶ correr(app)  ──▶  scorer  ──▶  reporte
  app(prompt, modelo) ────┘                       (LLM-judge,
                                                   exact-match,
                                                   RAGAS…)

  → bloquea el merge si el score baja umbral X
```

**Outputs típicos:**
- Tabla por ejemplo (input, output, score, reasoning)
- Métrica agregada (avg, p50/p95)
- Diff vs versión anterior
- **Verde/Rojo** para CI


## Diseño de eval datasets — parte 1: golden manual

**Un golden dataset bien hecho cuesta más que el prototipo.** Y vale más.

**Criterios de selección:**
- **Cobertura** — todas las intenciones reales (no sólo las "fáciles").
- **Estratificación** — proporción real por categoría/dificultad/idioma.
- **Edge cases** — preguntas ambiguas, fuera de dominio, adversariales.
- **Diversidad léxica** — paráfrasis, errores tipográficos, variaciones formales/informales.
- **Frescura** — datos no contaminados con el training del modelo.

**Tamaño mínimo (regla de campo):**

| Etapa | Tamaño |
|---|---|
| Smoke test (pre-PR) | 20-50 ejemplos |
| Regresión semanal | 200-500 |
| Acceptance pre-release | 500-2000 |

> No empezar con 5000. Empezar con 30 *bien elegidos*, expandir cuando duele.


## Diseño de eval datasets — parte 2: sintético + rotación

**Generación sintética con LLMs** (cuando manual no escala):

```python
prompt = f'''Generá 10 preguntas variadas sobre el dominio "{tema}".
Reglas: 3 fáciles, 4 medias, 3 difíciles. 2 de las difíciles deben ser
ambiguas o fuera de dominio. Devolvé JSON: [{{q, gold, dificultad}}].'''
```

**Riesgos del sintético:**
- **Sesgo del generador** — si lo arma GPT-4, evals con GPT-4 quedan fáciles.
- **Distribución artificial** — no refleja queries reales.
- **Contaminación** — el modelo a evaluar puede haber visto outputs similares.

**Mitigación:** mezclar **30% golden manual + 70% sintético**, validar con humanos por muestreo, **rotar** el dataset cada N iteraciones para evitar overfitting al benchmark interno.


## LLM-as-judge — diseño del juez

**Estructura mínima del prompt:**

```
SYSTEM:
  Sos un evaluador experto. Tu única tarea es scorear la respuesta
  contra la rúbrica y devolver JSON estricto.

USER:
  [Pregunta]: ...
  [Respuesta a evaluar]: ...
  [Contexto / referencia ideal (opcional)]: ...

  Rúbrica:
  - Faithfulness (0-5): ¿afirmaciones soportadas por el contexto?
  - Relevance (0-5): ¿responde la pregunta?
  - Completeness (0-5): ¿cubre los aspectos pedidos?

  Devolvé JSON: {scores: {...}, reasoning: "..."}
```

**Tips de diseño:**
- **Rúbrica explícita** > "evalualo bien". Cada dimensión por separado.
- **Output estructurado** (JSON) — parseable, audit-friendly.
- **Reasoning antes que score** (CoT) — calibra mejor.
- **Modelo del juez ≠ modelo evaluado** cuando se pueda (evita auto-bias).
- **Few-shot** del juez con casos límite ya scoreados a mano.

> 📖 *Zheng et al. (2023). Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* https://arxiv.org/abs/2306.05685


## LLM-as-judge — calibración y sesgos

**Sesgos conocidos del juez:**

| Sesgo | Qué hace | Mitigación |
|---|---|---|
| **Position bias** | Prefiere la primera (o segunda) respuesta en pairwise | Aleatorizar orden, evaluar ambas posiciones y promediar |
| **Length bias** | Prefiere respuestas largas | Penalizar length, o normalizar por tokens |
| **Self-preference** | Si el juez es GPT-4, prefiere outputs de GPT-4 | Cross-model evaluation (3+ jueces) |
| **Authority bias** | Prefiere respuestas con tono formal/seguro | Rúbrica con criterio de "evidencia, no certeza" |
| **Verbosity = quality** | Más palabras = mejor para el juez | Métricas separadas de calidad y concisión |

**Calibración contra humanos:**
1. Tomar 50-100 ejemplos.
2. Scorear con humanos (idealmente 2-3 anotadores → IAA con Cohen's κ).
3. Scorear con el juez.
4. Reportar correlación (Spearman / Pearson). **Aceptable si κ_humano-juez > 0.5.**

**Pairwise vs pointwise:**
- **Pointwise** (score 0-5 por respuesta): más informativo, menos confiable.
- **Pairwise** ("A es mejor que B"): más robusto, escalable a leaderboards (Chatbot Arena).


## RAGAS — métricas y fórmulas

```
  Pipeline:  Query → Retrieval → Chunks → LLM → Respuesta
                       │                    │
              ┌────────┴────────┐  ┌────────┴────────┐
              ▼                 ▼  ▼                 ▼
         Context Precision   Context Recall  Faithfulness  Answer Relevance
         "¿chunks         "¿están todos    "¿afirmaciones   "¿la respuesta
          relevantes        los chunks      soportadas?"     responde?"
          al principio?"    necesarios?"
```

| Métrica | Fórmula simplificada | Rango |
|---|---|---|
| **Faithfulness** | `# afirmaciones soportadas / # afirmaciones totales` | 0-1 |
| **Answer Relevance** | Similitud entre query original y queries reconstruidas desde la respuesta | 0-1 |
| **Context Precision** | `mean(precision@k)` ponderada por posición de chunks relevantes | 0-1 |
| **Context Recall** | `# afirmaciones del gold cubiertas por chunks / # afirmaciones del gold` | 0-1 |

**Ejemplo numérico (Faithfulness):**
- Respuesta: "Los modelos se despliegan como microservicios con Docker."
- 2 afirmaciones: ["se despliegan como microservicios", "se usan con Docker"].
- Chunks dicen sólo lo primero. → `Faithfulness = 1/2 = 0.5`.

> 📖 *Es et al. (2023). RAGAS.* https://arxiv.org/abs/2309.15217


## RAGAS — casos límite y cuándo falla

**Cuándo RAGAS no es suficiente:**

| Caso | Problema | Qué hacer |
|---|---|---|
| Pregunta multi-hop ("X y por qué") | Recall del retrieval baja → métrica engañosa | Eval por sub-pregunta, o usar BEIR |
| Respuestas creativas o subjetivas | Faithfulness se vuelve meaningless | Reemplazar por LLM-as-judge custom |
| Idiomas no-inglés | El juez (GPT-4) puede tener performance dispar | Validar con anotadores nativos |
| Preguntas que NO deberían responderse | RAGAS no penaliza "alucinar abstención" | Agregar métrica de "knew_to_refuse" |
| Modelo del juez muy parecido al evaluado | Auto-preference bias | Rotar jueces, o usar humanos por muestreo |

**Costo oculto:** RAGAS llama al LLM **varias veces por ejemplo** (faithfulness solo: 1 call por afirmación). En 1000 ejemplos × 5 afirmaciones promedio = **5000 calls al juez**. Presupuestar.

> RAGAS es excelente como **señal**. No es ground truth.


## Safety evals — jailbreak, toxicity, bias

**Más allá de calidad: ¿es seguro deployear?**

| Categoría | Qué se mide | Datasets / herramientas |
|---|---|---|
| **Jailbreak resistance** | ¿El modelo cede ante prompts adversariales? | AdvBench, HarmBench, JailbreakBench |
| **Toxicity** | ¿Genera contenido tóxico, ofensivo, hateful? | Perspective API, RealToxicityPrompts, ToxiGen |
| **Bias (demográfico)** | ¿Trata distinto por género/raza/edad? | BBQ, BOLD, StereoSet, CrowS-Pairs |
| **PII leakage** | ¿Reproduce datos personales del training? | Custom canary strings, MEMIT-style probes |
| **Hallucination factual** | Inventa hechos contraverificables | TruthfulQA, FActScore, HaluEval |

**Cuándo es obligatorio:** producto con usuarios externos, dominio regulado (salud, legal, financiero), menores como audiencia, cualquier integración con redes sociales.

**Cuándo es nice-to-have:** prototipos internos, herramientas dev, audiencias técnicas.

> No es opcional para producto. Lo que es opcional es **cuán formal** es el proceso.


## Red-teaming methodology

**Manual red-teaming:**
1. Reclutar 3-5 adversarios (idealmente fuera del equipo).
2. Brief: "rompé el modelo, documentá cada hallazgo".
3. Categorizar por severidad (CVSS-style: alto/medio/bajo).
4. Iterar: fix → re-test.

**Automated red-teaming (escala):**

```
  ┌──────────────────┐         ┌──────────────────┐
  │  Attacker LLM    │ ──────▶ │  Target LLM      │
  │  (genera        │         │  (el que         │
  │   prompts        │ ◀────── │   evaluamos)     │
  │   adversariales) │  refusal │                  │
  └──────────────────┘  / leak  └──────────────────┘
           ▲                              │
           │                              ▼
           └────── feedback loop ──── judge LLM
                  (clasifica si      (¿el output viola
                  hubo bypass)        la policy?)
```

**Frameworks notables:**
- **PAIR** (Chao et al. 2023) — attacker iterativo
- **GPTFuzzer** — fuzzing genético sobre seed prompts
- **HarmBench** — pipeline + dataset estandarizado

> 📖 *Chao et al. (2023). Jailbreaking Black Box LLMs in Twenty Queries.* https://arxiv.org/abs/2310.08419

> Output esperado: tabla de **attack success rate** por categoría, con ejemplos de prompts que pasaron.


## Lectura crítica de benchmarks públicos

| Benchmark | Mide | Limitación principal |
|---|---|---|
| **MMLU** | Conocimiento multitarea (57 dominios) | Contaminación masiva en modelos 2024+ |
| **GSM8K** | Razonamiento matemático (escolar) | Ya saturado (>95% en SOTA) |
| **HumanEval** | Generación de código Python | 164 ejemplos, gameable |
| **BIG-Bench Hard** | Razonamiento difícil | Heterogéneo, difícil de interpretar agregado |
| **MT-Bench** | Conversación multi-turn (LLM-judge) | Sólo 80 preguntas, sesgo del juez (GPT-4) |
| **Chatbot Arena** | Preferencia humana pairwise | Refleja preferencias de la comunidad LMSYS, no usuarios finales |
| **HELM** | Holistic (acc + fairness + robustness + cost) | Pesado, costoso de correr |
| **HumanEval+ / SWE-Bench** | Código real (issues GitHub) | Contaminación + complejidad de setup |

**Reglas de oro:**
1. **Un benchmark mide una proxy, no tu producto.** Buen MMLU ≠ buen asistente.
2. **Sospecha de contamination.** Si salió antes del cutoff del modelo, revisalo.
3. **Mirá la variance**, no sólo la mean. Diferencias <1pt suelen ser ruido.
4. **Combiná benchmarks.** Ningún número solo decide.


## Armar tu propio benchmark de dominio

**Protocolo de 5 pasos:**

```
  1. Definir tarea           → input, output esperado, restricciones
                               (ej. "clasificar tickets de soporte
                                en {bug, feature, account, billing}")

  2. Recolectar inputs       → 200-500 reales (anonimizados)
                               + 50-100 sintéticos (edge cases)

  3. Labelear gold           → 2-3 anotadores, IAA via Cohen's κ
                               κ < 0.6 → la rúbrica está mal

  4. Elegir métrica          → exact_match (clasificación)
                               + LLM-judge (calidad explicación)
                               + custom (compliance, etc.)

  5. Validar reproducibilidad → correr 3 veces, varianza < 2pt
                               (con seed fijo y temperature=0)
```

**Output entregable:**
- `benchmark_v1.jsonl` — el dataset versionado
- `eval_protocol.md` — métricas, modelos, condiciones
- `baseline_results.md` — score de modelos conocidos como referencia
- **Versionado en git como cualquier otro asset.**

> El benchmark **es el contrato** entre el equipo y el producto. Sin él, "mejoramos el modelo" no significa nada.


## Frameworks de eval offline — landscape

| Herramienta | Sweet spot | Open Source | Notas |
|---|---|---|---|
| **Promptfoo** | A/B de prompts en CLI/CI, regression suites | Sí (MIT) | Excelente DX en YAML, integra con CI fácil |
| **DeepEval** | Pytest-style con métricas built-in (RAGAS-like) | Sí | Buena para "test-driven prompts" |
| **OpenAI Evals** | Estándar de evals declarativos (graders YAML) | Sí | Repo grande, comunidad activa |
| **Inspect AI (UK AISI)** | Evals científicas, agentic tasks, sandboxing | Sí | Bueno para safety / capabilities research |
| **Arize Phoenix** | Tracing + evals offline + UI local | Sí | Lo que Arize AX es en cloud, pero local |
| **LangSmith** | Eval + tracing tightly integrado con LangChain | No | Bueno si ya estás en LangChain |
| **Langfuse** | Tracing + evals + prompt mgmt | Sí (self-host) | Alternativa OSS a LangSmith |

> **No elijas la herramienta primero.** Elegí la métrica primero, después la herramienta que mejor la implementa.


# Bloque 3
## Deploy / release — patrones de puesta en producción

---

*El gap entre "pasó la eval offline" y "está en manos de usuarios reales".*


## Eval online — A/B, shadow, canary

```
  A/B testing                Shadow mode                Canary
  ───────────                ───────────                ──────
  ┌────────┐                  ┌────────┐                ┌────────┐
  │ 50% v1 │                  │  v1    │ ──▶ usuario   │ 5%  v2 │ ──▶ usuario
  │ 50% v2 │ ──▶ usuario      │  v2    │ ──▶ /dev/null │ 95% v1 │ ──▶ usuario
  └────────┘                  └────────┘   (compara)   └────────┘
  Compará métricas            Sin riesgo,              Rollout gradual,
  online (CTR,                pero no medís            rollback fácil
  retention)                  preferencia              ante anomalía
```

| Patrón | Mide | Riesgo | Cuándo usar |
|---|---|---|---|
| **A/B test** | Preferencia / KPI de negocio | Medio (50% expuesto a v2) | Hipótesis clara y métrica online |
| **Shadow mode** | Performance técnica (latency, error rate) | Bajo (no afecta UX) | Cambio de modelo / arquitectura |
| **Canary** | Regresión amplia (todo a la vez) | Bajo a medio | Releases generales |

**Métricas online típicas para LLMs:**
- Tasa de respuesta del usuario (clic, copy, follow-up)
- Tiempo de la sesión, # mensajes, abandono
- Thumbs up/down explícito
- Re-asks ("eso no era lo que pregunté")
- p95 latency, error rate, $ por trace


## Prompt versioning & experimentación

**Anti-pattern típico:** prompt hardcodeado en el código, cambia entre PRs sin tracking.

**Solución mínima — prompt registry:**

```python
# prompts/qa_assistant_v3.txt
SYSTEM:
  Sos un asistente que responde basándose SÓLO en el contexto provisto.
  Si no sabés, decí "no lo sé" en vez de inventar.

# código
prompt = registry.get('qa_assistant', version='v3')
log_event('prompt_used', version='v3', trace_id=current_trace())
```

**Lo que tu sistema debe permitir:**
- Versionar prompts como código (git, semver: `v1.2.0`).
- Linkear cada trace a la versión usada (`prompt_version` como atributo).
- Rollback inmediato (cambiar default version, no necesita deploy).
- **Regression suite obligatoria** antes de promover una versión:
  - Mismo dataset offline → score >= versión anterior.
  - Mismas safety evals → no regresión.

> El prompt **es código**. Los prompts sin versionar son la deuda técnica más insidiosa de los sistemas LLM.


## Experiment tracking en LLMOps

**Para cada experimento (o release), trackear:**

| Atributo | Por qué importa |
|---|---|
| `prompt_version` | Reproducibilidad post-mortem |
| `model` + `model_version` | Mismo "GPT-4" tiene snapshots distintos |
| `temperature`, `top_p`, `max_tokens` | Cambios silenciosos de comportamiento |
| `dataset_hash` (offline) | Saber sobre qué datos se midió |
| `eval_scores` (todos) | Comparar runs, detectar regresión |
| `git_commit` del código | Reproducibilidad determinística |
| `cost_total`, `latency_p50/p95` | Restricciones operacionales |
| `timestamp`, `user` | Auditoría |

**Tooling:**
- **MLflow / Weights & Biases** — adaptados a LLM (W&B Prompts, MLflow LLM evaluate).
- **Arize AX / LangSmith / Langfuse** — runs como first-class concept.
- **DIY:** un JSONL por run, indexado en SQLite/DuckDB. Funciona y es portable.

> La regla: **si no puedo reproducir un run de hace 3 meses, no tengo experiment tracking**. Tengo un log.


## LLMOps lifecycle — model registry & deployment patterns

```
  ┌────────────┐    ┌────────────┐    ┌────────────┐    ┌────────────┐
  │  Experiment│───▶│   Model    │───▶│  Staging   │───▶│ Production │
  │  Tracking  │    │  Registry  │    │ (eval gate)│    │ (canary→100%)│
  └────────────┘    └────────────┘    └────────────┘    └────────────┘
       │                  │                 │                  │
       ▼                  ▼                 ▼                  ▼
   N runs por           1 source of      regression           rollback
   feature              truth, semver    suite + sign-off     automático
                        (prompt + model + params              ante regresión
                         como un "modelo")                    de eval online
```

**Deployment patterns aplicables a LLMs:**
- **Blue/Green** — dos entornos paralelos, switch instantáneo, doble costo.
- **Canary progresivo** — 1% → 5% → 25% → 100% con auto-rollback si algún SLO se viola.
- **Feature flags por usuario/segmento** — control fino, A/B nativo.
- **Shadow + Canary híbrido** — shadow primero (sin riesgo), después canary (con tráfico).

**Auto-rollback triggers típicos:**
- Drop > X% en eval online (LLM-judge sobre tráfico real)
- Spike > Y% en latency p95 / error rate
- Drop > Z% en feedback positivo
- Spike de safety incidents (toxicity, jailbreak success)

> El registro y el rollback automático son lo que separa "experimento puesto en prod" de "producto operable".


# Bloque 4
## Producción / monitoring — cuando ya hay tráfico real

---

*Lo que importa no es lo que pasa el día del deploy. Es lo que pasa **3 meses después**.*


## Qué loggear en producción

<img src="figures/monitoring_pipeline.svg" alt="Pipeline App → LLM → Logger middleware → Storage → Dashboard. Abajo, los 4 datos clave: input completo, output completo, latencia/costo, señal de calidad" class="slide-figure" style="width: 86%;"/>

**Las 4 cosas mínimas (por trace):**

| Categoría | Atributos |
|---|---|
| **Conversacional** | `input` completo (con system prompt), `output` completo, `messages` history |
| **Operacional** | `latency_total`, `latency_per_span`, `tokens_in`, `tokens_out`, `model`, `prompt_version` |
| **Económico** | `cost_usd` (calculado o reportado), `cache_hit` (si aplica) |
| **Calidad** | `eval_score` (LLM-judge async), `user_feedback` (thumbs, copy, re-ask) |

**Privacidad / compliance:**
- **PII redaction** antes de loggear (regex + LLM detector).
- Retención configurada por categoría (logs operacionales 30d; conversacionales 7d).
- Tracing distribuido respetando GDPR / consentimiento.

> Sin logs no hay debugging post-mortem. Sin atributos estandarizados, los logs son ruido.


## Cost & latency observability

**Anatomía de costo de una request:**

```
  Una llamada típica RAG (~$0.025):
  ─────────────────────────────────
  embed(query)         ~$0.0001     (cheap)
  retrieve (k=10)      ~0 (local)
  rerank (cross-enc)   ~$0.001      (~30 spans, gpu)
  generate (LLM)       ~$0.024      (98% del costo)

  → para optimizar costo: 95% del impacto está en el LLM call.
```

**Levers de optimización (de mayor a menor impacto):**

| Lever | Impacto típico | Tradeoff |
|---|---|---|
| **Cache de respuestas** (hash de query + context) | 30-70% en queries repetidas | Latencia primera vez sigue mala |
| **Model routing** (smaller model para queries simples) | 50-80% en mix | Calidad heterogénea, hay que monitorearla |
| **Prompt compression** | 20-40% (Microsoft LLMLingua) | Setup y mantenimiento |
| **Batching** (donde aplica) | Hasta 5x throughput | No reduce $ por request, sí latencia agregada |
| **Quantization** (self-hosted) | 2-4x throughput | Calidad puede caer |

**SLOs típicos en producción:**
- p95 latency < 3s (Q&A simple), < 10s (RAG complejo)
- p99 < 2x p95
- error rate < 0.5%
- cost per query < límite por endpoint


## Drift detection

<img src="figures/drift_detection.svg" alt="Cuatro tipos de drift: input drift, quality drift, cost drift, abuse drift. Abajo, un mini-dashboard mostrando accuracy bajando y costo subiendo a lo largo de 3 meses, con alerta disparada" class="slide-figure" style="width: 86%;"/>

**Los 4 tipos de drift en LLM apps:**

| Tipo | Síntoma | Detección | Causa típica |
|---|---|---|---|
| **Input drift** | Distribución de queries cambia | Embedding distance vs baseline | Cambio de producto, nuevo segmento de usuarios |
| **Quality drift** | Eval score baja sin cambio de versión | LLM-judge online sobre sample | Modelo upstream cambió silenciosamente |
| **Cost drift** | $/query sube | Métrica directa | Queries más largas, menos cache hits |
| **Abuse drift** | Spike de safety violations | Toxicity classifier en logs | Usuarios adversariales, nuevo ataque |

**Alertas razonables (umbrales de campo):**
- Input drift: distancia coseno > 0.15 vs baseline → review humana.
- Quality drift: avg LLM-judge cae > 5% vs 7 días → page.
- Cost drift: $/query sube > 20% week-over-week → page.
- Abuse drift: tasa de safety viol. > 0.5% del tráfico → page.

> Tu sistema fue bueno el día del deploy. Tres meses después, no necesariamente — y el modelo upstream pudo haber cambiado **sin que te avisen**.


## Feedback loops en producción

**Señales explícitas vs implícitas:**

| Tipo | Ejemplos | Pros | Contras |
|---|---|---|---|
| **Explícitas** | thumbs up/down, rating 1-5, "¿esto te sirvió?" | Intencionales, auditables | Bajo opt-in (1-5% típico), sesgo de extremos |
| **Implícitas** | Re-ask, copy del output, follow-up question, abandono, tiempo en respuesta | Cobertura completa | Ruidosas, requieren modeling |

**Loop típico: feedback → dataset → eval → mejora**

```
  thumbs_down   ──┐
  re-ask        ──┼──▶ "casos negativos"  ──▶  agregar al
  abandono      ──┘                              golden dataset
                                                       │
                                                       ▼
                                                 eval offline en
                                                 próxima iteración
                                                       │
                                                       ▼
                                                 mejorar prompt /
                                                 modelo / retrieval
```

**Human-in-the-loop selectivo:**
- 100% review si: jailbreak detectado, score juez < 2, cost outlier.
- 1-5% sample aleatorio para calibración del LLM-judge.

> El feedback loop es lo que **convierte usuarios en data**. Sin él, mejorás a ciegas.


# Bloque 5
## Cierre — el loop continuo

---

*Cómo se enchufan offline, online, monitoring y feedback en un ciclo de mejora.*


## El loop completo — prompt → producto → mejora

<img src="figures/feedback_loop.svg" alt="Ciclo de 6 nodos: diseño → eval offline → producción → eval online → análisis → feedback → vuelve al diseño. En el centro: PROMPT LIFECYCLE" class="slide-figure" style="width: 86%;"/>

```
   ┌──────────────────────────────────────────────────────────┐
   │                                                            │
   │   diseño  →  eval offline  →  staging  →  canary deploy   │
   │      ▲          (golden)        (eval gate)        │       │
   │      │                                              ▼       │
   │   feedback ◀── eval online ◀── traces ◀── 100% prod        │
   │   (humano +    (LLM-judge      (Arize/Phoenix              │
   │    señales      sobre tráfico    + drift detection)         │
   │    impl.)       muestreado)                                  │
   │                                                            │
   └──────────────────────────────────────────────────────────┘
```

> Igual que el código, **el prompt nunca está "terminado"**. Evoluciona con feedback de la realidad.

> La diferencia con ML clásico: el ciclo es **continuo y mensual**, no "modelo entrenado una vez al año". Cada release puede degradar (o mejorar) silenciosamente.


## ¿Cuándo invertir en cada cosa?

**No todos los proyectos necesitan todo desde día 1.**

| Madurez | Mínimo viable | Nice-to-have | Skippeable |
|---|---|---|---|
| **Prototipo / Demo** | Smoke test (10 ejemplos), logs básicos | LLM-judge manual | Drift, A/B, registry |
| **Beta / Early adopters** | Golden dataset (50-100), tracing básico, thumbs feedback | RAGAS / safety evals, dashboards | Auto-rollback, red-teaming formal |
| **Producción usuarios externos** | Todo lo de beta + safety evals + canary deploy + cost monitoring | Red-team trimestral, multi-judge | (mucho ya es must-have) |
| **Producto regulado / escala** | + SLOs auditables, retention compliance, audit trail completo, eval gate en CI | Continuous red-team, formal verification | (poco) |

**Antipattern común:** invertir en dashboards y registry antes de tener un golden dataset. Empezá por el dataset, lo demás se construye encima.

**El "primer eval" pragmático:**
1. Día 1 — 20 ejemplos a mano, exact match o regex donde puedas.
2. Día 7 — 50 ejemplos, LLM-judge para los abiertos.
3. Día 30 — primer dashboard de tracing.
4. Día 90 — drift detection + canary.


## 🛠 Práctica end-to-end con Arize AX

**Lo que vamos a hacer (35 min):**

```
  ┌─────────────────────────────────────────────────────────┐
  │ 1. Crear cuenta gratis en https://app.arize.com/        │
  │ 2. Levantar un chatbot Q&A sobre IA/ML (pre-built)      │
  │ 3. Correr eval offline con LLM-as-judge sobre 20 Q&A    │
  │ 4. Instrumentar tracing con OpenTelemetry → Arize       │
  │ 5. Generar tráfico y ver dashboards online              │
  │ 6. Simular drift y ver la alerta                         │
  │ 7. Reflexión final: ¿qué medirías en TU próximo proyecto?│
  └─────────────────────────────────────────────────────────┘
```

**Notebook:** [`clase03/notebooks/01_arize_eval_handson.ipynb`](notebooks/01_arize_eval_handson.ipynb)

**Lo que necesitás antes de empezar:**
- Cuenta de Groq (ya la tenés de Clase 2).
- Cuenta de Arize gratuita (signup en clase, ~3 min).
- Python con `arize`, `arize-otel`, `openinference-instrumentation-openai` instalado.

> Demo del docente sobre la pipeline RAG primero (5 min) → después ejercicio individual con el chatbot (30 min).


## Lo que vimos hoy

| Bloque | Concepto clave | Conexión con el resto del curso |
|---|---|---|
| B1 — Fundamentos | Eval de LLMs ≠ eval de ML clásico; reference-based vs free; tracing/observability como base | Antes de elegir herramienta, decidir qué medir |
| B2 — Pre-deploy | Golden datasets, LLM-as-judge (diseño + sesgos), RAGAS (deep dive), safety + red-team, benchmarks propios, landscape | Lo que corre en CI; bloquea merge si baja umbral |
| B3 — Deploy | A/B, shadow, canary; prompt versioning; experiment tracking; model registry; auto-rollback | La diferencia entre "experimento" y "producto operable" |
| B4 — Producción | Logging, cost & latency, drift detection (4 tipos), feedback loops explícitos vs implícitos | Lo que ven tus dashboards reales |
| B5 — Cierre | El loop completo; cuándo invertir en cada cosa según madurez | Pragmática: no todo proyecto necesita todo desde día 1 |

**Stack que probamos hoy:**
```
arize · arize-otel · openinference-instrumentation-groq · groq · pandas · matplotlib
```

**Lo que sigue siendo válido para Clase 4 (Agentes):**
- Todo este bloque aplica también a agentes — **más todavía**, porque los agentes son sistemas multi-call mucho más difíciles de evaluar que un LLM call único.
- `llamar_llm()` + tracing + LLM-as-judge → las herramientas no cambian; cambia la **unidad de evaluación** (trayectoria del agente, no respuesta única).


---

## 📚 Bibliografía

### Papers fundamentales — Evaluación y benchmarks
- Es, S. et al. (2023). *RAGAS — Automated Evaluation of Retrieval Augmented Generation.* EMNLP. https://arxiv.org/abs/2309.15217
- Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* https://arxiv.org/abs/2306.05685
- Manakul, P. et al. (2023). *SelfCheckGPT — Zero-Resource Black-Box Hallucination Detection.* https://arxiv.org/abs/2303.08896
- Liang, P. et al. (2023). *HELM — Holistic Evaluation of Language Models.* TMLR. https://arxiv.org/abs/2211.09110
- Chao, P. et al. (2023). *Jailbreaking Black Box LLMs in Twenty Queries (PAIR).* https://arxiv.org/abs/2310.08419
- Wang, B. et al. (2024). *DecodingTrust — Comprehensive Trustworthiness Evaluation of GPT Models.* https://arxiv.org/abs/2306.11698

### Documentación / herramientas
- **Arize AX** — https://docs.arize.com/
- **Arize Phoenix** (OSS local) — https://docs.arize.com/phoenix
- **OpenTelemetry para Python** — https://opentelemetry.io/docs/instrumentation/python/
- **OpenInference** (semantic conventions LLM) — https://github.com/Arize-ai/openinference
- **RAGAS docs** — https://docs.ragas.io/
- **Promptfoo** — https://www.promptfoo.dev/
- **Inspect AI** (UK AISI) — https://inspect.ai-safety-institute.org.uk/

### Lectura complementaria
- Karpathy, A. (2024). *Intro to Large Language Models* (charla). https://www.youtube.com/watch?v=zjkBMFhNj_g — el final aborda monitoring y safety.
- Anthropic (2024). *Responsible Scaling Policy.* — marco de evaluación de capacidades y safety.
- OpenAI (2024). *Preparedness Framework.* — categorías de riesgo y umbrales de eval.


## Clase 4 — ¿qué viene?

Hoy aprendimos a **medir y operar** sistemas LLM. En Clase 4 le damos al modelo **capacidad de actuar**.

---

- **¿Qué es un agente?** — percibir, razonar, actuar, observar, iterar.
- **Patrón ReAct** — razonamiento + acciones intercalados.
- **Tool calling y MCP** — cómo el LLM decide qué herramienta usar.
- **Sistemas multiagente** — orquestador + agentes especializados.
- **TP Integrador** — agente RAG funcional + instrumentado (junta todo: clases 1-3b).

> Lo que vimos hoy es **load-bearing** para clase 4: un agente sin tracing y sin evals es indebuggable.
